# Notebook 2 — Hotspot Detection and LST Map

Computes median Land Surface Temperature (LST) from Landsat 8/9 for the hot
season (March–May, 2022–2024) over Ouagadougou, identifies thermal hotspots
using a statistical threshold (mean + 1σ), and produces an interactive map.

**Prerequisites:**
- Authenticated Google Earth Engine account
- GEE asset: `projects/ee-helyne/assets/Ouaga_boundary`

**Outputs:**
- `figures/map_hotspots_median_lst_2022-2024.html` (interactive Leaflet map)
- `figures/pub/lst_hotspot_map.png` (static publication figure)

**Context:**
In Ouagadougou, the surrounding rural landscape is bare laterite soil and dry
brush which heats up faster than the city during the day, creating a daytime
"cool island" effect (Lindén, 2011). Our hotspot classification is therefore
intra-urban (mean + 1σ within the administrative boundary) rather than
urban–rural contrast.

In [ ]:
import ee
import geemap
import geemap.colormaps as cm
from IPython.display import Image

ee.Initialize()

## Load study area boundary

In [ ]:
urban_fc = ee.FeatureCollection("projects/ee-helyne/assets/Ouaga_boundary")
urban = urban_fc.geometry()

## Helper function: Landsat LST processing

In [ ]:
def process_landsat(img):
    """Compute LST in Celsius with QA-based cloud masking.

    Applies Collection 2 Level 2 scale factors to ST_B10 and masks
    pixels where the QA_PIXEL 'clear' bit (bit 6) is not set.
    """
    lst_kelvin = img.select("ST_B10").multiply(0.00341802).add(149.0)
    lst_celsius = lst_kelvin.subtract(273.15).rename("LST")

    qa = img.select("QA_PIXEL")
    mask = qa.bitwiseAnd(1 << 6).neq(0)

    return lst_celsius.updateMask(mask).copyProperties(img, ["system:time_start"])

## Load and process Landsat imagery

In [ ]:
# Time period: hot season (March-May) across 2022-2024
START_YEAR = 2022
START_MONTH, END_MONTH = "03", "05"

l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

landsat_collection = (
    l8.merge(l9)
    .filterBounds(urban)
    .filterDate(f"{START_YEAR}-{START_MONTH}-01", f"2024-{END_MONTH}-30")
    .filter(ee.Filter.calendarRange(3, 5, "month"))
    .filter(ee.Filter.lt("CLOUD_COVER", 20))
    .map(process_landsat)
)

print(f"Collection size: {landsat_collection.size().getInfo()} images")

## Compute median LST and hotspot threshold

In [ ]:
lst_median = landsat_collection.median().clip(urban)

# City-wide statistics
stats = lst_median.reduceRegion(
    reducer=ee.Reducer.mean().combine(
        reducer2=ee.Reducer.stdDev(), sharedInputs=True
    ),
    geometry=urban,
    scale=30,
    maxPixels=1e13,
    bestEffort=True,
)

stats_local = stats.getInfo()
mean_lst = stats_local["LST_mean"]
std_lst = stats_local["LST_stdDev"]
threshold = mean_lst + std_lst

print(f"Mean LST: {mean_lst:.2f} \u00b0C")
print(f"Std Dev:  {std_lst:.2f} \u00b0C")
print(f"Hotspot threshold (mean + 1\u03c3): {threshold:.2f} \u00b0C")

## Create binary hotspot classification

In [ ]:
hotspot_binary = (
    lst_median.select("LST")
    .gt(threshold)
    .rename("hotspot")
    .clip(urban)
    .updateMask(lst_median.select("LST").mask())
)

## Interactive map

In [ ]:
Map = geemap.Map()
Map.setCenter(-1.45, 12.345, 11)
Map.add_basemap("Esri.WorldImagery")

palette = cm.get_palette("inferno")
vis_params = {"min": 25, "max": 60, "palette": palette}

Map.addLayer(lst_median, vis_params, "Median LST (Hot Season)")
Map.add_colorbar(vis_params=vis_params, label="LST (\u00b0C)")

hotspot_vis = {"min": 0, "max": 1, "palette": ["32CD32", "FF2400"]}
Map.addLayer(hotspot_binary, hotspot_vis, "Hotspot Classification", False)
Map.add_legend(
    legend_dict={
        "Non-Hotspot": "32CD32",
        "Hotspot (LST > Mean + 1SD)": "FF2400",
    },
    title="UHI Classification",
    position="bottomright",
)

Map.addLayer(urban_fc.geometry(), {"color": "black"}, "Ouagadougou Boundary", False)
Map.add_text(
    "Ouagadougou LST (Mar-May 2022-2024)",
    position="bottomleft",
    font_size="16px",
    font_color="white",
)

Map

## Export interactive map to HTML

In [ ]:
Map.to_html("../figures/map_hotspots_median_lst_2022-2024.html", title="")
print("Saved: figures/map_hotspots_median_lst_2022-2024.html")

## Export static publication figure

Uses the GEE thumbnail API to produce a PNG suitable for the publication
composite figure.

In [ ]:
import urllib.request
from pathlib import Path

pub_dir = Path("..") / "figures" / "pub"
pub_dir.mkdir(parents=True, exist_ok=True)

thumb_url = lst_median.getThumbUrl({
    "min": 25,
    "max": 60,
    "palette": ["000004", "320A5A", "781C6D", "BC3754", "ED6925", "F9B641", "FCFFA4"],
    "dimensions": 1200,
    "region": urban,
})

out_path = pub_dir / "lst_hotspot_map.png"
urllib.request.urlretrieve(thumb_url, out_path)
print(f"Saved: {out_path}")

Image(url=thumb_url)